# Phân loại cảm xúc (Sentiment Classification) bình luận phim IMDb bằng BERT

> **Nguồn gốc & ghi công (credits)**
>
> - Bộ dữ liệu: **Large Movie Review Dataset (IMDb)** — Andrew L. Maas, Raymond E. Daly, Peter T. Pham, Dan Huang,
>   Andrew Y. Ng, Christopher Potts (2011), *"Learning Word Vectors for Sentiment Analysis"*, ACL 2011.
>   Trang gốc: <https://ai.stanford.edu/~amaas/data/sentiment/>. Nội dung bình luận thuộc về người viết trên IMDb —
>   **chỉ dùng cho mục đích nghiên cứu, học tập**.
> - Dữ liệu được tải qua bản phát hành chính thức trên Hugging Face Hub:
>   [`stanfordnlp/imdb`](https://huggingface.co/datasets/stanfordnlp/imdb) (giữ nguyên cách chia train/test gốc).
> - Mô hình: **BERT** — Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova (2019),
>   *"BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding"*, NAACL 2019.
>   Trọng số `bert-base-uncased` do Google phát hành (Apache 2.0), tải qua Hugging Face Hub.
> - Quy trình fine-tuning (tokenize → DataLoader → AdamW + warmup → vòng lặp huấn luyện) tham khảo
>   [Hugging Face LLM Course — Chương 3: Fine-tuning a pretrained model](https://huggingface.co/learn/llm-course/chapter3)
>   (Apache 2.0); siêu tham số theo khuyến nghị của bài báo BERT và của Sun và cộng sự (2019).
> - Thư viện: [🤗 Transformers](https://github.com/huggingface/transformers), [🤗 Datasets](https://github.com/huggingface/datasets),
>   [PyTorch](https://pytorch.org/), [scikit-learn](https://scikit-learn.org/).

## 0. Tổng quan — notebook này làm gì?

### Bài toán: Phân loại cảm xúc (Sentiment Classification)
Cho một **bình luận phim** (một đoạn văn bản tiếng Anh, thường dài vài trăm từ), ta muốn biết người viết
**khen** (`positive`) hay **chê** (`negative`) bộ phim. Đây là bài toán **phân loại văn bản nhị phân**:
đầu vào là cả một đoạn văn, đầu ra là **một** nhãn (khác với NER — mỗi từ một nhãn).

| Nhãn | Giá trị | Ví dụ (rút gọn) |
|---|---|---|
| `negative` | 0 | *"A waste of two hours. The plot makes no sense…"* |
| `positive` | 1 | *"One of the best films I've seen this year…"* |

### Hai mô hình được so sánh

| | **Baseline: TF-IDF + Logistic Regression** | **BERT fine-tuning** |
|---|---|---|
| Biểu diễn văn bản | "Túi từ" (bag-of-words): đếm từ/cụm 2 từ, **bỏ qua thứ tự** | Vector theo **ngữ cảnh**: mỗi từ được hiểu dựa trên các từ xung quanh |
| Tri thức sẵn có | Không — học từ con số 0 trên 22.500 bình luận | Đã **tiền huấn luyện** trên ~3,3 tỉ từ (BooksCorpus + Wikipedia) |
| Số tham số | Vài trăm nghìn – vài triệu trọng số tuyến tính | ~110 triệu |
| Chi phí | Huấn luyện vài chục giây trên CPU | Cần GPU; vài phút đến hàng giờ tuỳ phần cứng |
| Diễn giải | Dễ: đọc trực tiếp trọng số từng từ | Khó ("hộp đen") |

Câu hỏi chính của notebook: **BERT tốt hơn baseline bao nhiêu, ở những loại bình luận nào, và cái giá phải trả là gì?**

### Quy trình
1. Tải dữ liệu, tách **train → train/dev** (dev để chọn siêu tham số; **test chỉ dùng một lần** để báo cáo).
2. Phân tích dữ liệu (EDA).
3. Baseline TF-IDF + Logistic Regression: dò siêu tham số trên dev, đánh giá trên test, đọc trọng số.
4. Fine-tune BERT: tokenize, huấn luyện, chọn epoch tốt nhất theo dev, đánh giá trên test.
5. So sánh: độ đo, kiểm định thống kê McNemar, phân tích theo độ dài bình luận, phân tích lỗi.
6. Demo trên câu tự nhập.

### Độ đo
Lớp "dương" (positive class) là `positive` (nhãn 1).

- **Accuracy** = tỉ lệ bình luận được phân loại đúng. Dữ liệu IMDb **cân bằng 50/50** nên accuracy là độ đo chính
  (bộ phân loại đoán bừa chỉ đạt ~50%).
- **Precision** = trong số bình luận mô hình đoán là `positive`, bao nhiêu % thật sự `positive`.
- **Recall** = trong số bình luận thật sự `positive`, mô hình tìm ra bao nhiêu %.
- **F1** = trung bình điều hoà: $F1 = \dfrac{2 \cdot P \cdot R}{P + R}$.

Với dữ liệu cân bằng, accuracy và F1 thường rất gần nhau; precision và recall cho biết mô hình **lệch** về phía nào
(ví dụ precision cao nhưng recall thấp → mô hình "khó tính", hay đoán `negative`).

## 1. Cài đặt và nạp thư viện

| Thư viện | Vai trò |
|---|---|
| `datasets` | Tải bộ dữ liệu IMDb từ Hugging Face Hub; xử lý dữ liệu theo lô (`map`). |
| `transformers` | Tokenizer và mô hình BERT tiền huấn luyện; bộ lập lịch learning rate. |
| `torch` (PyTorch) | Tính toán tensor trên GPU, vòng lặp huấn luyện. |
| `scikit-learn` | TF-IDF, Logistic Regression, dò siêu tham số, các độ đo. |
| `scipy` | Kiểm định thống kê McNemar. |
| `pandas`, `matplotlib` | Bảng biểu và biểu đồ. |

**Phần cứng**: phần BERT nên chạy trên **GPU** (ví dụ Google Colab: *Runtime → Change runtime type → T4 GPU*).
Máy Mac chip Apple (M1/M2/M3…) dùng được GPU qua backend `mps` nhưng chậm hơn nhiều; chỉ có CPU thì nên
giảm kích thước dữ liệu bằng tham số `N_TRAIN` ở mục 5.1. Phần baseline chạy tốt trên CPU.

Nếu sau lần cài đầu tiên import bị lỗi, hãy **khởi động lại kernel** (*Kernel → Restart*) rồi chạy lại.

In [ ]:
%pip install -q "transformers>=4.40" "datasets>=2.18" torch "scikit-learn>=1.3" scipy pandas matplotlib

Nạp thư viện. Vài điểm đáng chú ý:
- `TfidfVectorizer` + `LogisticRegression` ghép thành một `Pipeline` → gọi `fit`/`predict` một lần cho cả hai bước.
- `PredefinedSplit`: giống notebook NER, ta dò siêu tham số trên **một cách chia train/dev cố định** thay vì cross-validation.
- `AutoTokenizer`, `AutoModelForSequenceClassification`: tự chọn đúng lớp tokenizer/mô hình theo tên (`bert-base-uncased`).
- `DataCollatorWithPadding`: ghép các bình luận thành lô và **đệm (pad)** cho bằng độ dài bình luận dài nhất **trong lô**
  (đệm động — nhanh hơn nhiều so với đệm mọi bình luận lên độ dài tối đa).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [ ]:
import os
import re
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import scipy.stats

import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

import torch
from torch.utils.data import DataLoader
import datasets
from datasets import load_dataset, Dataset
import transformers
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)
from tqdm.auto import tqdm

print('scikit-learn:', sklearn.__version__)
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('datasets    :', datasets.__version__)

### Cố định hạt giống ngẫu nhiên và chọn thiết bị

- `SEED = 42`: cố định mọi nguồn ngẫu nhiên (Python, NumPy, PyTorch) để chạy lại cho kết quả gần như giống nhau.
  (Trên GPU một số phép toán vẫn không hoàn toàn tất định nên con số có thể lệch ở chữ số thứ 3–4.)
- Thiết bị: ưu tiên `cuda` (GPU NVIDIA) → `mps` (GPU Apple) → `cpu`.

In [ ]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

if torch.cuda.is_available():
    device = torch.device('cuda')
    print('Thiết bị: GPU', torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Thiết bị: GPU Apple (mps)')
else:
    device = torch.device('cpu')
    print('Thiết bị: CPU — phần BERT sẽ rất chậm, nên đặt N_TRAIN nhỏ ở mục 5.1')

## 2. Tải và chuẩn bị bộ dữ liệu IMDb

Bộ dữ liệu gốc của Maas và cộng sự (2011) gồm **50.000 bình luận có nhãn** từ IMDb, chia sẵn **25.000 train / 25.000 test**,
mỗi tập cân bằng 12.500 `positive` / 12.500 `negative`. Ngoài ra có 50.000 bình luận **không nhãn** (`unsupervised`)
dùng cho học không giám sát — notebook này **không dùng** tới.

Cách gán nhãn của tác giả: dựa vào **số sao** người viết chấm (thang 10): **≤ 4 sao → `negative`**, **≥ 7 sao → `positive`**;
bình luận 5–6 sao (trung tính) bị **loại bỏ**. Mỗi bộ phim có tối đa 30 bình luận, và **phim trong train không xuất hiện trong test**
→ mô hình không thể "học thuộc" tên phim/diễn viên để đoán.

`load_dataset('stanfordnlp/imdb')` tải bản chính thức (~85 MB) về thư mục `data/imdb/` (cạnh notebook); lần sau đọc lại từ đó.

In [ ]:
# [DATA-LOAD]
DATA_DIR = os.path.join('data', 'imdb')
imdb = load_dataset('stanfordnlp/imdb', cache_dir=DATA_DIR)
imdb

Mỗi mẫu có hai trường: `text` (bình luận) và `label` (0 = `negative`, 1 = `positive`).
Ta chuyển sang `pandas.DataFrame` cho dễ phân tích.

In [ ]:
LABEL_NAMES = ['negative', 'positive']

train_full_df = imdb['train'].to_pandas()
test_df = imdb['test'].to_pandas()
print(imdb['train'].features)
train_full_df.head()

### Tách tập phát triển (dev)

IMDb chỉ có train và test. Nếu chọn siêu tham số (hệ số `C` của Logistic Regression, epoch tốt nhất của BERT)
bằng chính tập test thì kết quả cuối sẽ **lạc quan giả tạo** ("nhìn trộm" đáp án). Vì vậy ta tách **10% train làm dev**:

- `stratify=label`: giữ tỉ lệ 50/50 trong cả train và dev;
- `random_state=SEED`: cách chia cố định.

Kết quả: **22.500 train / 2.500 dev / 25.000 test**. Cả hai mô hình đều chỉ học trên train, chọn tham số trên dev, và được
đánh giá **một lần** trên test — so sánh công bằng.

*Lưu ý*: cách chia ngẫu nhiên này khiến một số phim có bình luận ở cả train lẫn dev (khác với test), nên điểm trên dev
có thể **nhỉnh hơn** điểm trên test một chút.

In [ ]:
train_df, dev_df = train_test_split(
    train_full_df, test_size=0.1, stratify=train_full_df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
data = {'train': train_df, 'dev': dev_df, 'test': test_df}
for split, df in data.items():
    print(f'{split:5s}: {len(df):6d} bình luận')

## 3. Phân tích bộ dữ liệu (EDA)

### 3.1. Phân bố nhãn

In [ ]:
print(f"{'Tập':6s} {'negative':>9s} {'positive':>9s} {'% pos':>7s}")
for split, df in data.items():
    counts = df['label'].value_counts().sort_index()
    print(f'{split:6s} {counts[0]:9d} {counts[1]:9d} {100 * counts[1] / len(df):6.1f}%')

Cả ba tập đều **cân bằng tuyệt đối** → accuracy là độ đo đáng tin, không cần kỹ thuật xử lý mất cân bằng (class weight, oversampling…).

### 3.2. Xem vài bình luận mẫu
In 600 ký tự đầu của một bình luận mỗi loại. Để ý chuỗi `<br /><br />` — thẻ xuống dòng HTML còn sót lại khi thu thập dữ liệu.

In [ ]:
for label in [0, 1]:
    sample = train_df[train_df['label'] == label].iloc[0]['text']
    print(f'=== {LABEL_NAMES[label]} ===')
    print(sample[:600], '...\n')

### 3.3. Độ dài bình luận

Độ dài quyết định trực tiếp chi phí của BERT (chi phí self-attention tăng theo **bình phương** độ dài chuỗi) và
giới hạn **512 token** của BERT. Ta đếm số từ (tách theo khoảng trắng) của mỗi bình luận.

In [ ]:
for df in data.values():
    df['n_words'] = df['text'].str.split().str.len()

print(train_df.groupby('label')['n_words'].describe().round(0).rename(index=dict(enumerate(LABEL_NAMES))))
print()
for threshold in [200, 400]:
    print(f'Train: {100 * (train_df["n_words"] > threshold).mean():.1f}% bình luận dài hơn {threshold} từ')

fig, ax = plt.subplots(figsize=(10, 4))
for label in [0, 1]:
    ax.hist(train_df.loc[train_df['label'] == label, 'n_words'].clip(upper=1200),
            bins=60, alpha=0.6, label=LABEL_NAMES[label])
ax.set_xlabel('Số từ trong bình luận (cắt ở 1200)')
ax.set_ylabel('Số bình luận')
ax.set_title('Phân bố độ dài bình luận (train)')
ax.legend()
plt.show()

Nhận xét: phân bố **lệch phải** — đa số bình luận dài 100–300 từ nhưng có một "đuôi" bình luận rất dài (hơn 1.000 từ).
Độ dài **gần như không khác nhau** giữa hai nhãn → độ dài bản thân nó không phải tín hiệu để đoán cảm xúc.

### 3.4. Nhiễu HTML

In [ ]:
has_br = train_df['text'].str.contains('<br', regex=False)
print(f'{100 * has_br.mean():.1f}% bình luận train chứa thẻ <br />')

Hơn một nửa bình luận chứa `<br />`. Nếu giữ nguyên, TF-IDF sẽ coi `br` là một "từ" xuất hiện cực nhiều,
còn tokenizer BERT sẽ tốn token cho `<`, `br`, `/`, `>`. Ta viết hàm `clean_text` thay thẻ này bằng khoảng trắng
và dùng **chung** cho cả hai mô hình (để so sánh công bằng). Các bước tiền xử lý khác (viết thường, tách từ)
do từng mô hình tự đảm nhiệm.

In [ ]:
def clean_text(text):
    # Thay thẻ <br />, <br/>, <br> bằng khoảng trắng; gộp nhiều khoảng trắng liên tiếp
    text = re.sub(r'<br\s*/?>', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

for df in data.values():
    df['text'] = df['text'].map(clean_text)

print(train_df['text'].iloc[0][:300])

### 3.5. Từ phổ biến theo từng nhãn

Đếm các từ xuất hiện nhiều nhất trong mỗi nhãn (đã bỏ **stop words** tiếng Anh như *the, and, is*).

In [ ]:
counter_vec = CountVectorizer(stop_words='english', min_df=5)
counts = counter_vec.fit_transform(train_df['text'])
vocab = np.array(counter_vec.get_feature_names_out())

word_freq = {}
for label in [0, 1]:
    freq = np.asarray(counts[(train_df['label'] == label).values].sum(axis=0)).ravel()
    word_freq[label] = freq
    top = vocab[np.argsort(-freq)[:15]]
    print(f'{LABEL_NAMES[label]:8s}:', ', '.join(top))

Hai danh sách **gần như trùng nhau** (*movie, film, like, good, just, story…*): từ phổ biến nhất không phân biệt được cảm xúc.
Ngay cả *good* cũng xuất hiện nhiều trong bình luận chê (*"not good"*, *"good actors wasted on a bad script"*).
Thứ cần tìm là những từ **chênh lệch** tần suất giữa hai nhãn. Một cách đơn giản là **log-odds ratio**:

$$\text{score}(w) = \log \frac{(c_{\text{pos}}(w) + 1) / N_{\text{pos}}}{(c_{\text{neg}}(w) + 1) / N_{\text{neg}}}$$

($c$ = số lần xuất hiện, $N$ = tổng số từ của nhãn; cộng 1 để tránh chia cho 0). Ta chỉ xét từ xuất hiện ít nhất 200 lần.

In [ ]:
n_pos, n_neg = word_freq[1].sum(), word_freq[0].sum()
log_odds = np.log((word_freq[1] + 1) / n_pos) - np.log((word_freq[0] + 1) / n_neg)
frequent = (word_freq[0] + word_freq[1]) >= 200

order = np.argsort(log_odds[frequent])
print('Nghiêng về negative:', ', '.join(vocab[frequent][order[:15]]))
print('Nghiêng về positive:', ', '.join(vocab[frequent][order[::-1][:15]]))

Các từ "cực đoan" như *worst, waste, awful, poorly* (chê) và *excellent, superb, wonderful, perfect* (khen) có tính
phân biệt rất cao. Đây chính là thứ mà baseline TF-IDF + Logistic Regression sẽ học: **gán trọng số cho từng từ**.
Điểm yếu của cách này: không hiểu **phủ định** (*"not bad"*), **tương phản** (*"great cast, terrible movie"*) hay **mỉa mai**.

### Tóm tắt phân tích

| Đặc điểm | Hệ quả cho mô hình |
|---|---|
| Nhãn cân bằng 50/50 | Accuracy là độ đo chính; không cần xử lý mất cân bằng. |
| Bình luận dài (trung vị ~170 từ, có bình luận > 1.000 từ) | BERT phải **cắt bớt** (giới hạn 512 token); chi phí BERT cao. |
| Nhiễu HTML `<br />` | Làm sạch trước khi đưa vào cả hai mô hình. |
| Từ phổ biến giống nhau giữa hai nhãn | Cần trọng số kiểu TF-IDF/hồi quy, không chỉ đếm tần suất. |
| Nhiều từ mang cảm xúc mạnh (*worst, excellent*) | Baseline túi từ sẽ khá mạnh — IMDb vốn là bài toán "dễ" cho mô hình tuyến tính. |
| Phủ định, tương phản, mỉa mai | Nơi kỳ vọng BERT (hiểu ngữ cảnh) vượt trội baseline. |

## 4. Baseline: TF-IDF + Logistic Regression

### TF-IDF (Term Frequency – Inverse Document Frequency)
Mỗi bình luận được biểu diễn bằng một vector rất dài (mỗi chiều ứng với một từ hoặc cụm từ trong từ điển):

$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t), \qquad \text{idf}(t) = \log\frac{1 + N}{1 + \text{df}(t)} + 1$$

- $\text{tf}(t, d)$: số lần từ $t$ xuất hiện trong bình luận $d$;
- $\text{df}(t)$: số bình luận chứa $t$; $N$: tổng số bình luận.
  Từ xuất hiện ở **mọi nơi** (như *movie*) có idf thấp → bị giảm trọng số; từ **hiếm, đặc trưng** có idf cao.
- Vector được chuẩn hoá về độ dài 1 (chuẩn L2) để bình luận dài và ngắn so sánh được.

Cấu hình `TfidfVectorizer`:

| Tham số | Giá trị | Ý nghĩa |
|---|---|---|
| `ngram_range` | dò `(1,1)` hoặc `(1,2)` | `(1,2)` = dùng cả **từ đơn** và **cụm 2 từ** (bigram) → bắt được *"not good"*, *"must see"*. |
| `min_df` | `2` | Bỏ từ chỉ xuất hiện ở 1 bình luận (thường là lỗi chính tả, tên riêng) → từ điển gọn hơn. |
| `max_df` | `0.9` | Bỏ từ xuất hiện ở hơn 90% bình luận (gần như không mang thông tin). |
| `sublinear_tf` | `True` | Dùng $1 + \log(\text{tf})$ thay cho $\text{tf}$: từ lặp lại 10 lần không quan trọng gấp 10 lần lặp 1 lần. |

### Logistic Regression
Mô hình tuyến tính: $P(\text{positive} \mid d) = \sigma(\mathbf{w}^\top \mathbf{x}_d + b)$, với $\sigma(z) = 1/(1+e^{-z})$.
Mỗi từ có **một trọng số** $w_t$: dương → đẩy về `positive`, âm → đẩy về `negative`.
Tham số `C` là **nghịch đảo độ mạnh điều chuẩn L2**: `C` nhỏ → phạt trọng số lớn mạnh hơn (chống quá khớp);
`C` lớn → mô hình bám sát dữ liệu train hơn.

### Dò siêu tham số trên dev
Giống notebook NER: ghép train + dev, dùng `PredefinedSplit` (`-1` = luôn để huấn luyện, `0` = dùng để chấm điểm)
để mỗi tổ hợp chỉ cần **huấn luyện một lần trên train, chấm accuracy trên dev**. Lưới tìm kiếm nhỏ (2 × 6 = 12 tổ hợp)
nên dùng `GridSearchCV` (thử hết). Mất khoảng 1–3 phút tuỳ máy.

In [ ]:
%%time
baseline = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, max_df=0.9, sublinear_tf=True, strip_accents='unicode')),
    ('clf', LogisticRegression(max_iter=1000, solver='liblinear')),
])
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.5, 1, 2, 4, 8, 16],
}

X_train_dev = pd.concat([train_df['text'], dev_df['text']], ignore_index=True)
y_train_dev = pd.concat([train_df['label'], dev_df['label']], ignore_index=True)
dev_split = PredefinedSplit([-1] * len(train_df) + [0] * len(dev_df))

gs = GridSearchCV(baseline, param_grid, cv=dev_split, scoring='accuracy',
                  n_jobs=-1, verbose=1, refit=False)
gs.fit(X_train_dev, y_train_dev)
print('best params :', gs.best_params_)
print('best dev acc:', round(gs.best_score_, 4))

Bảng accuracy trên dev của từng tổ hợp: hàng = kiểu n-gram, cột = `C`.

In [ ]:
grid_results = pd.DataFrame(gs.cv_results_['params'])
grid_results['dev_accuracy'] = gs.cv_results_['mean_test_score']
grid_results['tfidf__ngram_range'] = grid_results['tfidf__ngram_range'].astype(str)
grid_results.pivot(index='tfidf__ngram_range', columns='clf__C', values='dev_accuracy').round(4)

Thường thấy: **thêm bigram tăng accuracy** rõ rệt (bắt được *"not good"*, *"waste of"*…), và `C` khá lớn (ít điều chuẩn)
cho kết quả tốt vì dữ liệu nhiều, đặc trưng thưa.

### Huấn luyện baseline cuối cùng trên train
Dùng tổ hợp tốt nhất, huấn luyện lại trên **train** (không gồm dev — để công bằng với BERT) và đo thời gian.

In [ ]:
baseline.set_params(**gs.best_params_)

start = time.perf_counter()
baseline.fit(train_df['text'], train_df['label'])
lr_train_time = time.perf_counter() - start

start = time.perf_counter()
lr_test_proba = baseline.predict_proba(test_df['text'])[:, 1]
lr_infer_time = time.perf_counter() - start
lr_test_pred = (lr_test_proba >= 0.5).astype(int)

n_features = len(baseline.named_steps['tfidf'].vocabulary_)
print(f'Kích thước từ điển : {n_features:,} đặc trưng (từ + cụm 2 từ)')
print(f'Thời gian huấn luyện: {lr_train_time:.1f} giây')
print(f'Thời gian dự đoán 25.000 bình luận test: {lr_infer_time:.1f} giây')

### Đánh giá baseline trên tập test

Hàm `compute_metrics` tính 4 độ đo với lớp dương là `positive`; `classification_report` in thêm số liệu cho **từng lớp**
(`support` = số bình luận thật của lớp đó). Kết quả được lưu vào dict `results` để so sánh ở mục 7.

In [ ]:
def compute_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1)
    return {'accuracy': accuracy_score(y_true, y_pred), 'precision': precision, 'recall': recall, 'f1': f1}


def report(name, y_true, y_pred):
    m = compute_metrics(y_true, y_pred)
    print(f'[{name}]  ' + '  '.join(f'{k} = {v:.4f}' for k, v in m.items()))
    print()
    print(classification_report(y_true, y_pred, target_names=LABEL_NAMES, digits=4))
    return m


y_test = test_df['label'].values
results = {}
results['TF-IDF + LR'] = report('TF-IDF + LR / test', y_test, lr_test_pred)

**Ma trận nhầm lẫn (confusion matrix)**: hàng = nhãn thật, cột = nhãn dự đoán. Đường chéo chính là số dự đoán đúng;
ô góc trên phải = bình luận chê bị đoán là khen (false positive), ô góc dưới trái = khen bị đoán là chê (false negative).

In [ ]:
def plot_confusion(y_true, y_pred, title, ax):
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=LABEL_NAMES).plot(
        ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(title)
    ax.grid(False)

fig, ax = plt.subplots(figsize=(4.5, 4))
plot_confusion(y_test, lr_test_pred, 'TF-IDF + LR (test)', ax)
plt.show()

### Baseline đã học được gì?

Ưu điểm lớn của mô hình tuyến tính là **dễ diễn giải** (giống trọng số CRF trong notebook NER):
mỗi đặc trưng (từ hoặc cụm 2 từ) có một trọng số. In 20 đặc trưng có trọng số dương nhất và âm nhất.

In [ ]:
feature_names = baseline.named_steps['tfidf'].get_feature_names_out()
coefs = baseline.named_steps['clf'].coef_.ravel()
order = np.argsort(coefs)

print(f"{'Ủng hộ NEGATIVE':28s} {'Ủng hộ POSITIVE':28s}")
for neg_i, pos_i in zip(order[:20], order[::-1][:20]):
    print(f'{coefs[neg_i]:7.2f}  {feature_names[neg_i]:18s} {coefs[pos_i]:7.2f}  {feature_names[pos_i]:18s}')

Một số nhận xét (đối chiếu với log-odds ở mục 3.5):

- Các từ cảm xúc mạnh (*worst, awful, waste, boring* ↔ *great, excellent, perfect, best*) đứng đầu — đúng như kỳ vọng.
- Một số bigram lọt vào danh sách: *"the worst"*, *"waste of"* (chê), *"the best"*, *"well worth"* (khen) —
  bigram giúp mô hình bắt được một phần ngữ cảnh.
- Có cả những từ **không mang cảm xúc** khi đứng riêng, như *nothing*, *even* (chê) và *still*, *bit* (khen).
  Chúng phản ánh **cách người ta viết**: *"nothing happens"*, *"even the actors couldn't save it"* so với
  *"still worth watching"*, *"a bit slow but…"* (lời chê nhẹ thường đi kèm một bình luận tích cực).
- Bigram vẫn chỉ nhìn **2 từ liền nhau**: *"not good"* được học, nhưng *"not particularly good"* hay
  *"I wouldn't say it was good"* thì không. Đây là chỗ BERT có lợi thế.

Thử xem trọng số của một vài cụm phủ định cụ thể:

In [ ]:
vocab_index = baseline.named_steps['tfidf'].vocabulary_
for phrase in ['good', 'not good', 'bad', 'not bad', 'funny', 'not funny', 'recommend', 'not recommend']:
    weight = coefs[vocab_index[phrase]] if phrase in vocab_index else float('nan')
    print(f'{phrase:15s} {weight:7.2f}')

## 5. Fine-tune BERT

### BERT là gì?
**BERT** (Bidirectional Encoder Representations from Transformers) là mạng **Transformer encoder** (12 lớp, 768 chiều,
12 đầu attention, ~110 triệu tham số với bản `base`), được **tiền huấn luyện** trên lượng văn bản khổng lồ bằng hai nhiệm vụ tự giám sát:

1. **Masked Language Modeling**: che ngẫu nhiên 15% từ và bắt mô hình đoán lại từ bị che dựa vào **cả hai phía** ngữ cảnh;
2. **Next Sentence Prediction**: đoán câu B có phải câu tiếp theo của câu A không.

Nhờ đó, mỗi từ được biểu diễn bằng một vector **phụ thuộc ngữ cảnh**: *"good"* trong *"not good"* khác *"good"* trong *"very good"*.

### Fine-tuning cho phân loại
- Đầu mỗi chuỗi có token đặc biệt **`[CLS]`**; vector đầu ra tại vị trí `[CLS]` được xem là biểu diễn **cả đoạn văn**.
- `AutoModelForSequenceClassification` gắn thêm một **lớp tuyến tính** (768 → 2) lên trên vector `[CLS]`.
  Lớp này **khởi tạo ngẫu nhiên** (sẽ có cảnh báo *"Some weights … newly initialized"* — đó là bình thường).
- Fine-tuning = huấn luyện **toàn bộ** mạng (cả 110 triệu tham số) với learning rate **rất nhỏ** trong vài epoch,
  để điều chỉnh nhẹ tri thức sẵn có cho bài toán của ta.

### 5.1. Cấu hình

| Tham số | Giá trị | Ý nghĩa |
|---|---|---|
| `MODEL_NAME` | `bert-base-uncased` | BERT bản base, **không phân biệt hoa thường** (tokenizer tự viết thường). |
| `MAX_LEN` | `256` | Số token tối đa mỗi bình luận; dài hơn sẽ bị **cắt bớt** (xem mục 5.2). Tối đa là 512. |
| `BATCH_SIZE` | `16` | Số bình luận mỗi lô. Hết bộ nhớ GPU thì giảm xuống 8. |
| `EPOCHS` | `2` | Số lần duyệt hết tập train. Bài báo BERT khuyến nghị 2–4. |
| `LEARNING_RATE` | `2e-5` | Khuyến nghị trong khoảng {2e-5, 3e-5, 5e-5}. Lớn hơn dễ làm "quên" tri thức tiền huấn luyện. |
| `WARMUP_RATIO` | `0.1` | 10% số bước đầu tăng dần learning rate từ 0 → giá trị đặt, sau đó giảm tuyến tính về 0. |
| `WEIGHT_DECAY` | `0.01` | Điều chuẩn L2 kiểu AdamW (không áp cho bias và LayerNorm). |
| `N_TRAIN` | `None` | `None` = dùng toàn bộ 22.500 bình luận train. Đặt số nhỏ (ví dụ `2000`) để chạy thử nhanh trên CPU/Mac. |

**Thời gian tham khảo** (toàn bộ train, `MAX_LEN = 256`, 2 epoch): GPU T4 trên Colab khoảng 30–40 phút (có dùng mixed precision);
GPU Apple (`mps`) có thể mất vài giờ; CPU không thực tế nếu không giảm `N_TRAIN`.
Dùng `MAX_LEN = 512` thường tăng accuracy thêm khoảng 0,5–1 điểm nhưng chậm hơn ~2–3 lần.

In [ ]:
# [BERT-CONFIG]
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 2
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
N_TRAIN = None      # ví dụ 2000 để chạy thử nhanh; None = toàn bộ train
N_EVAL = None       # giới hạn số bình luận dev/test khi chạy thử; None = toàn bộ

USE_AMP = device.type == 'cuda'   # mixed precision (fp16) chỉ bật trên GPU NVIDIA

### 5.2. Tokenizer: WordPiece

BERT không làm việc với "từ" theo nghĩa thông thường mà với **subword** (từ con) theo thuật toán **WordPiece**,
từ điển ~30.000 mục. Từ phổ biến giữ nguyên; từ hiếm bị tách thành nhiều mảnh, mảnh tiếp nối có tiền tố `##`.
Nhờ vậy BERT **không bao giờ gặp từ lạ** (out-of-vocabulary) — khác với TF-IDF (bỏ qua mọi từ không có trong từ điển).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example = "The cinematography was breathtaking, but the screenplay was unwatchable."
encoded = tokenizer(example)
print(tokenizer.convert_ids_to_tokens(encoded['input_ids']))
print('input_ids     :', encoded['input_ids'])
print('attention_mask:', encoded['attention_mask'])

- `[CLS]` và `[SEP]` được tự động thêm vào đầu và cuối chuỗi.
- `input_ids`: chỉ số của từng token trong từ điển; `attention_mask`: 1 = token thật, 0 = token đệm (`[PAD]`, xuất hiện khi ghép lô).
- Các từ hiếm như *cinematography*, *unwatchable* bị tách thành nhiều mảnh `##…`.

### Bao nhiêu bình luận vượt giới hạn?
Đếm số token (theo WordPiece) của toàn bộ bình luận train. Số token thường **nhiều hơn số từ** khoảng 20–30%
(dấu câu và từ hiếm tách thành nhiều token).

In [ ]:
train_token_lens = np.array([len(ids) for ids in tokenizer(train_df['text'].tolist())['input_ids']])
print(f'Số token: trung vị {np.median(train_token_lens):.0f}, trung bình {train_token_lens.mean():.0f}, '
      f'lớn nhất {train_token_lens.max()}')
for limit in [256, 512]:
    print(f'{100 * (train_token_lens > limit).mean():5.1f}% bình luận dài hơn {limit} token')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.clip(train_token_lens, 0, 1500), bins=60)
for limit in [MAX_LEN, 512]:
    ax.axvline(limit, color='black', linestyle='--')
    ax.text(limit + 10, ax.get_ylim()[1] * 0.9, f'{limit}', fontsize=10)
ax.set_xlabel('Số token WordPiece (cắt ở 1500)')
ax.set_ylabel('Số bình luận')
ax.set_title('Độ dài bình luận theo token (train)')
plt.show()

Với `MAX_LEN = 256`, một phần đáng kể bình luận bị cắt (chỉ giữ **phần đầu**). Thường thì cảm xúc đã lộ rõ ngay phần đầu
nên mất mát không lớn, nhưng với bình luận mà người viết **chốt ý ở cuối** (*"…Overall, I'd give it 3/10"*) thì mô hình
không thấy được. Sun và cộng sự (2019) cho thấy chiến lược **"đầu + cuối"** (giữ 128 token đầu và 382 token cuối)
tốt hơn chỉ giữ phần đầu — một hướng cải tiến để thử.

### 5.3. Chuẩn bị dữ liệu cho PyTorch

1. Chuyển DataFrame thành `datasets.Dataset`, tokenize theo lô bằng `map(batched=True)` (nhanh nhờ tokenizer viết bằng Rust),
   cắt bớt ở `MAX_LEN`; **chưa đệm** ở bước này.
2. Đổi tên cột `label` → `labels` (tên mà mô hình Hugging Face mong đợi để tự tính hàm mất mát).
3. `DataLoader` + `DataCollatorWithPadding`: mỗi lô được đệm tới độ dài bình luận dài nhất **trong lô** (đệm động).
   Tập train được **xáo trộn** (`shuffle=True`) mỗi epoch; dev/test giữ nguyên thứ tự để ghép lại với nhãn.

In [ ]:
def to_hf_dataset(df, n=None):
    if n is not None:
        df = df.sample(n=min(n, len(df)), random_state=SEED)
    ds = Dataset.from_pandas(df[['text', 'label']].reset_index(drop=True), preserve_index=False)
    ds = ds.map(lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=['text'])
    return ds.rename_column('label', 'labels'), df   # df giữ chỉ số gốc để ghép với baseline


train_ds, bert_train_df = to_hf_dataset(train_df, N_TRAIN)
dev_ds, bert_dev_df = to_hf_dataset(dev_df, N_EVAL)
test_ds, bert_test_df = to_hf_dataset(test_df, N_EVAL)

collator = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
                          generator=torch.Generator().manual_seed(SEED))
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE * 2, collate_fn=collator)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, collate_fn=collator)

print(train_ds)
batch = next(iter(train_loader))
print({k: tuple(v.shape) for k, v in batch.items()})

Mỗi lô là một dict tensor: `input_ids`, `token_type_ids` (luôn bằng 0 vì chỉ có một đoạn văn), `attention_mask`
có kích thước `(BATCH_SIZE, độ dài lớn nhất trong lô)`, và `labels` kích thước `(BATCH_SIZE,)`.

### 5.4. Khởi tạo mô hình, optimizer và bộ lập lịch learning rate

- `num_labels=2`, `id2label`/`label2id`: để mô hình biết tên nhãn (hiển thị đẹp khi lưu/tải lại).
- **AdamW**: biến thể của Adam với weight decay "tách rời" — optimizer chuẩn cho Transformer. Không áp weight decay cho bias và LayerNorm.
- **Linear schedule with warmup**: learning rate tăng tuyến tính từ 0 trong 10% số bước đầu (tránh các bước đầu "giật mạnh"
  khi lớp phân loại còn ngẫu nhiên), rồi giảm tuyến tính về 0.

In [ ]:
set_seed()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label=dict(enumerate(LABEL_NAMES)), label2id={name: i for i, name in enumerate(LABEL_NAMES)},
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Số tham số: {n_params / 1e6:.1f} triệu')

no_decay = ['bias', 'LayerNorm.weight']
optimizer = torch.optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
], lr=LEARNING_RATE)

num_training_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(WARMUP_RATIO * num_training_steps), num_training_steps=num_training_steps)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
print(f'Tổng số bước huấn luyện: {num_training_steps} ({len(train_loader)} bước/epoch × {EPOCHS} epoch)')

### 5.5. Vòng lặp huấn luyện

Mỗi bước (một lô):
1. **Lan truyền xuôi** (forward): mô hình trả về `logits` (điểm chưa chuẩn hoá cho 2 lớp) và `loss` (cross-entropy, vì ta đưa `labels` vào);
2. **Lan truyền ngược** (backward): tính gradient;
3. **Cắt gradient** (gradient clipping) ở chuẩn 1.0 để tránh bước cập nhật quá lớn;
4. Cập nhật trọng số (`optimizer.step()`) và learning rate (`scheduler.step()`).

**Mixed precision** (`torch.autocast` + `GradScaler`, chỉ trên GPU NVIDIA): tính toán ở fp16 → nhanh hơn ~2 lần, tốn ít bộ nhớ hơn.

Cuối mỗi epoch, đánh giá trên **dev** và **giữ lại trọng số của epoch có accuracy dev cao nhất** (early stopping đơn giản) —
dev đóng vai trò chọn mô hình, đúng như với baseline.

In [ ]:
@torch.no_grad()
def predict_proba(model, loader):
    # Trả về P(positive) cho từng mẫu, theo đúng thứ tự của loader
    model.eval()
    probs = []
    for batch in tqdm(loader, desc='predict', leave=False):
        batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            logits = model(**batch).logits
        probs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu())
    return torch.cat(probs).numpy()

In [ ]:
history = {'step': [], 'loss': [], 'epoch_dev': []}
best_dev_acc, best_state = -1.0, None
y_dev_bert = bert_dev_df['label'].values

set_seed()
start = time.perf_counter()
global_step = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    progress = tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}')
    for batch in progress:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            loss = model(**batch).loss
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        global_step += 1
        history['step'].append(global_step)
        history['loss'].append(loss.item())
        progress.set_postfix(loss=f'{loss.item():.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    dev_pred = (predict_proba(model, dev_loader) >= 0.5).astype(int)
    dev_metrics = compute_metrics(y_dev_bert, dev_pred)
    history['epoch_dev'].append(dev_metrics)
    print(f'epoch {epoch}: ' + '  '.join(f'dev {k} = {v:.4f}' for k, v in dev_metrics.items()))

    if dev_metrics['accuracy'] > best_dev_acc:
        best_dev_acc = dev_metrics['accuracy']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch

bert_train_time = time.perf_counter() - start
model.load_state_dict(best_state)
print(f'\nThời gian huấn luyện: {bert_train_time / 60:.1f} phút; chọn epoch {best_epoch} (dev accuracy = {best_dev_acc:.4f})')

### Đường cong mất mát (loss)

Loss của từng lô dao động mạnh, nên ta vẽ thêm **trung bình trượt** 50 bước. Loss giảm nhanh trong vài trăm bước đầu
(lớp phân loại học được cách dùng vector `[CLS]`), sau đó giảm chậm. Đường kẻ dọc đánh dấu ranh giới giữa các epoch.

In [ ]:
loss_series = pd.Series(history['loss'], index=history['step'])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(loss_series, alpha=0.3, label='loss từng lô')
ax.plot(loss_series.rolling(50, min_periods=1).mean(), label='trung bình trượt 50 bước')
for e in range(1, EPOCHS):
    ax.axvline(e * len(train_loader), color='black', linestyle='--')
ax.set_xlabel('Bước huấn luyện')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Quá trình fine-tune BERT')
ax.legend()
plt.show()

pd.DataFrame(history['epoch_dev'], index=pd.Index(range(1, EPOCHS + 1), name='epoch')).round(4)

(Tuỳ chọn) Lưu mô hình đã fine-tune để dùng lại mà không cần huấn luyện lại — bỏ dấu `#` để chạy.
Thư mục nặng khoảng 440 MB, **không nên** đưa lên GitHub.

In [ ]:
# model.save_pretrained('models/bert-imdb'); tokenizer.save_pretrained('models/bert-imdb')
# Tải lại: AutoModelForSequenceClassification.from_pretrained('models/bert-imdb')

## 6. Đánh giá BERT trên tập test

Dự đoán **một lần** trên 25.000 bình luận test với trọng số của epoch tốt nhất, đo thời gian dự đoán.

In [ ]:
start = time.perf_counter()
bert_test_proba = predict_proba(model, test_loader)
bert_infer_time = time.perf_counter() - start
bert_test_pred = (bert_test_proba >= 0.5).astype(int)
y_test_bert = bert_test_df['label'].values

print(f'Thời gian dự đoán {len(y_test_bert):,} bình luận test: {bert_infer_time:.1f} giây')
results['BERT'] = report('BERT / test', y_test_bert, bert_test_pred)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
plot_confusion(y_test, lr_test_pred, 'TF-IDF + LR (test)', ax1)
plot_confusion(y_test_bert, bert_test_pred, 'BERT (test)', ax2)
plt.tight_layout()
plt.show()

## 7. So sánh hai mô hình

### 7.1. Bảng tổng hợp

Ngoài 4 độ đo chất lượng, ta so sánh cả **chi phí**: thời gian huấn luyện, thời gian dự đoán và số tham số.
(Số tham số của baseline = số đặc trưng TF-IDF + 1 hệ số chặn.)

*Nếu đã đặt `N_EVAL` để chạy thử, BERT được đánh giá trên một mẫu con của test nên con số chỉ mang tính tham khảo.*

In [ ]:
comparison = pd.DataFrame(results).T
comparison['train time (s)'] = [lr_train_time, bert_train_time]
comparison['test inference (s)'] = [lr_infer_time, bert_infer_time]
comparison['parameters'] = [n_features + 1, n_params]
comparison.style.format({'accuracy': '{:.4f}', 'precision': '{:.4f}', 'recall': '{:.4f}', 'f1': '{:.4f}',
                         'train time (s)': '{:,.1f}', 'test inference (s)': '{:,.1f}', 'parameters': '{:,.0f}'})

In [ ]:
metric_names = ['accuracy', 'precision', 'recall', 'f1']
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.35
for k, (name, m) in enumerate(results.items()):
    values = [m[metric] for metric in metric_names]
    bars = ax.bar([i + k * width for i in range(len(metric_names))], values, width, label=name)
    ax.bar_label(bars, fmt='%.3f', fontsize=9)
ax.set_xticks([i + width / 2 for i in range(len(metric_names))])
ax.set_xticklabels(metric_names)
ax.set_ylim(0.8, 1.0)
ax.set_title('So sánh trên tập test')
ax.legend(loc='lower right')
plt.show()

**Mốc tham khảo trong tài liệu** (cùng tập test chính thức 25.000 bình luận):

| Phương pháp | Accuracy |
|---|---|
| Maas và cộng sự (2011) — word vectors + bag-of-words | 88,9% |
| TF-IDF/bigram + mô hình tuyến tính (baseline kiểu notebook này) | thường ~89–91% |
| NBSVM bigram — Wang & Manning (2012) | 91,2% |
| BERT-base fine-tuning, 512 token (Sun và cộng sự, 2019) | ~94–95% |

Kết quả của ta nên rơi vào khoảng tương ứng; BERT với `MAX_LEN = 256` thường thấp hơn bản 512 token một chút.

### 7.2. Khác biệt có ý nghĩa thống kê không? — Kiểm định McNemar

Hai mô hình được đánh giá **trên cùng các bình luận**, nên không so sánh hai accuracy như hai con số độc lập được.
**Kiểm định McNemar** chỉ xét những bình luận mà hai mô hình **bất đồng**:

- $b$ = số bình luận baseline đúng, BERT sai;
- $c$ = số bình luận baseline sai, BERT đúng.

Giả thuyết $H_0$: hai mô hình tốt như nhau ⇔ một bình luận "bất đồng" có xác suất 50/50 thuộc về $b$ hay $c$.
Ta dùng **kiểm định nhị thức chính xác** (`scipy.stats.binomtest`). p-value rất nhỏ (< 0,05) ⇒ bác bỏ $H_0$, chênh lệch là thật
chứ không phải do may rủi.

In [ ]:
# Ghép dự đoán của baseline vào đúng các bình luận BERT đã dự đoán (theo chỉ số gốc của test_df)
compare_df = bert_test_df.copy()
compare_df['lr_proba'] = lr_test_proba[compare_df.index]
compare_df['lr_pred'] = lr_test_pred[compare_df.index]
compare_df['bert_proba'] = bert_test_proba
compare_df['bert_pred'] = bert_test_pred
compare_df['lr_ok'] = compare_df['lr_pred'] == compare_df['label']
compare_df['bert_ok'] = compare_df['bert_pred'] == compare_df['label']

table = pd.crosstab(compare_df['lr_ok'].map({True: 'LR đúng', False: 'LR sai'}),
                    compare_df['bert_ok'].map({True: 'BERT đúng', False: 'BERT sai'}))
print(table, '\n')

b = int((compare_df['lr_ok'] & ~compare_df['bert_ok']).sum())
c = int((~compare_df['lr_ok'] & compare_df['bert_ok']).sum())
test = scipy.stats.binomtest(min(b, c), b + c, p=0.5)
print(f'b (LR đúng, BERT sai) = {b}')
print(f'c (LR sai, BERT đúng) = {c}')
print(f'McNemar exact p-value = {test.pvalue:.2e}')

Bảng 2×2 cho thấy: phần lớn bình luận **cả hai cùng đúng** (những bình luận "dễ" có từ cảm xúc rõ ràng);
số bình luận chỉ BERT đúng ($c$) thường **lớn hơn nhiều** số bình luận chỉ baseline đúng ($b$) — nhưng $b$ không bằng 0:
baseline vẫn thắng BERT ở một số trường hợp (ví dụ bình luận dài mà BERT bị cắt mất phần kết).

### 7.3. Phân tích theo độ dài bình luận

Chia bình luận test thành 5 nhóm (mỗi nhóm 20% số bình luận) theo số từ, tính accuracy từng mô hình trong mỗi nhóm.
Câu hỏi: việc **cắt bớt** ở `MAX_LEN` có làm BERT kém đi với bình luận dài không?

In [ ]:
compare_df['n_words'] = compare_df['text'].str.split().str.len()
compare_df['length_bucket'] = pd.qcut(compare_df['n_words'], q=5)
by_length = compare_df.groupby('length_bucket', observed=True).agg(
    n=('label', 'size'), lr_acc=('lr_ok', 'mean'), bert_acc=('bert_ok', 'mean'))
print(by_length.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(by_length))
ax.plot(x, by_length['lr_acc'], marker='o', label='TF-IDF + LR')
ax.plot(x, by_length['bert_acc'], marker='o', label='BERT')
ax.set_xticks(list(x))
ax.set_xticklabels([f'{int(iv.left)}–{int(iv.right)}' for iv in by_length.index])
ax.set_xlabel('Số từ trong bình luận')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy theo độ dài bình luận (test)')
ax.legend()
plt.show()

Thường thấy: **cả hai** mô hình kém dần khi bình luận dài hơn (bình luận dài hay "khen chê lẫn lộn", kể lể nội dung phim).
Khoảng cách giữa BERT và baseline thường **lớn nhất ở bình luận ngắn – trung bình** (BERT đọc được toàn bộ) và **thu hẹp ở nhóm dài nhất**
(BERT chỉ thấy phần đầu, trong khi TF-IDF "đọc" hết). Đây là bằng chứng ủng hộ việc thử `MAX_LEN = 512` hoặc chiến lược "đầu + cuối".

### 7.4. Những bình luận mà hai mô hình bất đồng

In vài bình luận **ngắn** (để dễ đọc) mà chỉ một trong hai mô hình đoán đúng, kèm xác suất `positive` mỗi mô hình đưa ra.

In [ ]:
def show_examples(mask, title, n=4, max_chars=400):
    print(f'===== {title} =====')
    subset = compare_df[mask].sort_values('n_words').head(n)
    for _, row in subset.iterrows():
        print(f"[nhãn thật: {LABEL_NAMES[row['label']]}]  P(pos) LR = {row['lr_proba']:.2f}, BERT = {row['bert_proba']:.2f}")
        print('  ' + row['text'][:max_chars] + ('…' if len(row['text']) > max_chars else ''))
        print()

show_examples(~compare_df['lr_ok'] & compare_df['bert_ok'], 'Chỉ BERT đúng')
show_examples(compare_df['lr_ok'] & ~compare_df['bert_ok'], 'Chỉ baseline đúng')

Khi đọc các ví dụ "chỉ BERT đúng", hãy tìm những mẫu hình sau — đúng là những chỗ túi từ không nắm được:

- **Phủ định / giảm nhẹ**: *"not as bad as people say"*, *"I can't say I didn't enjoy it"*;
- **Tương phản**: liệt kê nhiều điểm tốt rồi kết luận *"but in the end it just doesn't work"*;
- **Từ "tốt" dùng trong ngữ cảnh chê**: *"good for a laugh, unintentionally"*.

Các ví dụ "chỉ baseline đúng" thường là bình luận **nhập nhằng** (người chấm 4 sao nhưng lời lẽ khá nhẹ nhàng),
hoặc có từ khoá rất mạnh mà BERT bị "đánh lừa" bởi ngữ cảnh xung quanh.

## 8. Phân tích lỗi của BERT

Những lỗi **tự tin nhất** (xác suất sai rất gần 0 hoặc 1) thường tiết lộ nhiều điều: hoặc là bình luận thật sự khó
(mỉa mai, khen chê lẫn lộn), hoặc là **nhãn bị nhiễu** (nhãn suy ra từ số sao không khớp với lời văn).

In [ ]:
errors = compare_df[~compare_df['bert_ok']].copy()
errors['confidence'] = np.where(errors['bert_pred'] == 1, errors['bert_proba'], 1 - errors['bert_proba'])
print(f'BERT sai {len(errors)} / {len(compare_df)} bình luận '
      f"({(errors['label'] == 0).sum()} chê bị đoán là khen, {(errors['label'] == 1).sum()} khen bị đoán là chê)\n")

for _, row in errors.sort_values('confidence', ascending=False).head(5).iterrows():
    print(f"[nhãn thật: {LABEL_NAMES[row['label']]}, BERT đoán {LABEL_NAMES[row['bert_pred']]} "
          f"với độ tin cậy {row['confidence']:.3f}]")
    print('  ' + row['text'][:500] + ('…' if len(row['text']) > 500 else ''))
    print()

**Độ tin cậy có đáng tin không?** Chia dự đoán của BERT theo mức độ tin cậy và xem accuracy từng nhóm.
Một mô hình "biết mình biết gì" sẽ có accuracy cao ở nhóm tin cậy cao và thấp hơn rõ ở nhóm tin cậy ~0,5–0,7.
Trong ứng dụng thực tế, các bình luận có độ tin cậy thấp có thể được chuyển cho **người duyệt** thay vì tự động gán nhãn.

In [ ]:
conf = np.maximum(compare_df['bert_proba'], 1 - compare_df['bert_proba'])
bins = pd.cut(conf, [0.5, 0.7, 0.9, 0.99, 1.0], include_lowest=True)
calib = compare_df.groupby(bins, observed=True).agg(n=('bert_ok', 'size'), accuracy=('bert_ok', 'mean'))
calib['% số bình luận'] = 100 * calib['n'] / calib['n'].sum()
calib.round(3)

## 9. Demo: phân loại câu tự nhập

Đưa cùng một câu vào hai mô hình và so sánh xác suất `positive`. Các câu mẫu được chọn để thử **phủ định**, **tương phản**
và **mỉa mai** — những hiện tượng ngôn ngữ mà mục 4 dự đoán là điểm yếu của túi từ.

In [ ]:
@torch.no_grad()
def bert_proba_texts(texts):
    model.eval()
    enc = tokenizer([clean_text(t) for t in texts], truncation=True, max_length=MAX_LEN,
                    padding=True, return_tensors='pt').to(device)
    return torch.softmax(model(**enc).logits.float(), dim=-1)[:, 1].cpu().numpy()


def demo_sentiment(texts):
    lr_p = baseline.predict_proba([clean_text(t) for t in texts])[:, 1]
    bert_p = bert_proba_texts(texts)
    print(f"{'P(pos) LR':>9s} {'P(pos) BERT':>11s}  Câu")
    for t, p1, p2 in zip(texts, lr_p, bert_p):
        print(f'{p1:9.2f} {p2:11.2f}  {t}')


SENTENCES = [
    "This movie was absolutely wonderful, I loved every minute of it.",
    "This movie was not good at all.",
    "Not bad at all! I was pleasantly surprised.",
    "I expected to hate it, but it turned out to be one of the best films of the year.",
    "Great cast, beautiful scenery, and yet the most boring two hours of my life.",
    "Oh great, another sequel nobody asked for. Just what we needed.",
    "The plot was predictable, but the acting was superb and I would watch it again.",
]
demo_sentiment(SENTENCES)

Tự thử: thêm câu của bạn vào `SENTENCES` (tiếng Anh) rồi chạy lại ô trên. Một vài điều thường quan sát được:

- Câu khen/chê thẳng thắn: cả hai mô hình đều đúng, rất tự tin.
- *"not good"*, *"not bad"*: baseline xử lý được nếu bigram đó có trong từ điển, nhưng xác suất thường **kém dứt khoát** hơn BERT.
- *"expected to hate it, but…"*, *"great cast… yet the most boring"*: baseline bị kéo bởi các từ riêng lẻ (*hate* / *great*),
  BERT nắm được ý chính nằm **sau** từ *but / yet*.
- **Mỉa mai** (*"Oh great, … just what we needed"*): khó cho cả hai — mô hình chỉ học được mỉa mai nếu dữ liệu huấn luyện có đủ ví dụ.

## 10. Kết luận

| Tiêu chí | TF-IDF + Logistic Regression | BERT fine-tuning |
|---|---|---|
| Chất lượng | Mạnh bất ngờ trên IMDb (~90%) nhờ nhiều từ cảm xúc rõ ràng | Cao hơn vài điểm phần trăm, khác biệt có ý nghĩa thống kê (McNemar) |
| Hiểu ngữ cảnh | Chỉ trong cửa sổ 2 từ (bigram) | Phủ định, tương phản, quan hệ xa trong câu |
| Bình luận dài | Đọc hết văn bản | Bị giới hạn 512 token (ở đây 256) → mất thông tin phần cuối |
| Chi phí | Huấn luyện vài chục giây trên CPU; dự đoán gần như tức thì | Cần GPU; huấn luyện hàng chục phút; dự đoán chậm hơn hàng trăm lần |
| Diễn giải | Đọc trực tiếp trọng số từng từ | Cần kỹ thuật riêng (attention, SHAP, Integrated Gradients…) |

**Bài học**: luôn bắt đầu bằng một baseline đơn giản. Trên IMDb, baseline đã giải quyết phần lớn bài toán; BERT mang lại
cải thiện rõ rệt nhưng với chi phí tính toán lớn hơn nhiều bậc. Lựa chọn mô hình phụ thuộc vào việc vài điểm phần trăm
đó có đáng với chi phí trong ứng dụng cụ thể hay không.

### Hướng mở rộng
1. `MAX_LEN = 512` hoặc chiến lược cắt **"đầu + cuối"** (Sun và cộng sự, 2019) cho bình luận dài.
2. Dò siêu tham số BERT (learning rate {2e-5, 3e-5, 5e-5}, số epoch 2–4) trên dev.
3. Mô hình nhẹ hơn như **DistilBERT** (nhanh hơn ~60%, giữ ~97% chất lượng) hoặc mạnh hơn như **RoBERTa**, **DeBERTa**.
4. Baseline mạnh hơn: **NBSVM** (Wang & Manning, 2012), thêm trigram.
5. Kiểm tra khả năng **chuyển miền**: mô hình học trên bình luận phim áp dụng cho bình luận sản phẩm/nhà hàng tốt đến đâu?

## Tài liệu tham khảo

1. Andrew L. Maas, Raymond E. Daly, Peter T. Pham, Dan Huang, Andrew Y. Ng, Christopher Potts (2011).
   *Learning Word Vectors for Sentiment Analysis.* ACL 2011. <https://aclanthology.org/P11-1015/> · dữ liệu: <https://ai.stanford.edu/~amaas/data/sentiment/>
2. Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova (2019). *BERT: Pre-training of Deep Bidirectional Transformers
   for Language Understanding.* NAACL 2019. <https://aclanthology.org/N19-1423/>
3. Chi Sun, Xipeng Qiu, Yige Xu, Xuanjing Huang (2019). *How to Fine-Tune BERT for Text Classification?* CCL 2019.
   <https://arxiv.org/abs/1905.05583>
4. Sida Wang, Christopher D. Manning (2012). *Baselines and Bigrams: Simple, Good Sentiment and Topic Classification.* ACL 2012.
   <https://aclanthology.org/P12-2018/>
5. Ilya Loshchilov, Frank Hutter (2019). *Decoupled Weight Decay Regularization (AdamW).* ICLR 2019. <https://arxiv.org/abs/1711.05101>
6. Thomas G. Dietterich (1998). *Approximate Statistical Tests for Comparing Supervised Classification Learning Algorithms.*
   Neural Computation 10(7) — cơ sở cho việc dùng kiểm định McNemar để so sánh hai bộ phân loại.
7. Thomas Wolf và cộng sự (2020). *Transformers: State-of-the-Art Natural Language Processing.* EMNLP 2020 (Demos).
   <https://aclanthology.org/2020.emnlp-demos.6/>
8. Quentin Lhoest và cộng sự (2021). *Datasets: A Community Library for Natural Language Processing.* EMNLP 2021 (Demos).
   <https://aclanthology.org/2021.emnlp-demo.21/>
9. Fabian Pedregosa và cộng sự (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12. <https://jmlr.org/papers/v12/pedregosa11a.html>
10. Hugging Face LLM Course — Chapter 3: Fine-tuning a pretrained model. <https://huggingface.co/learn/llm-course/chapter3>
11. Dataset card `stanfordnlp/imdb`: <https://huggingface.co/datasets/stanfordnlp/imdb>